# Session 03 — Draw a helpful face-framing guide

Today’s new skill: use drawings on a webcam frame to tell a person what to do.

OpenCV finds a face-shaped region and returns a box. We use that box to draw a target and give one clear response: READY, ADJUST, or FACE NOT FOUND.

This detects face position only. It does not identify who the person is.


## Choose your webcam route

- In Colab: use the browser webcam preview, then capture a real webcam frame.
- On your own computer: run the continuous live-webcam loop.

Both routes use the same face detector, alignment rule, and drawing code.


## Set up OpenCV once

Run the next cell once. It removes conflicting OpenCV packages and installs one supported version. When it finishes, restart the runtime before running the notebook from the top.


In [ ]:
# Run this setup cell once. Do not continue to the next cell yet.
from pathlib import Path
import subprocess
import sys

ready_file = Path('/content/session3_opencv_ready')

if not ready_file.exists():
    # OpenCV packages all use the same cv2 name. Keep exactly one.
    for package in ['cv2', 'opencv-python', 'opencv-python-headless', 'opencv-contrib-python']:
        subprocess.run(
            [sys.executable, '-m', 'pip', 'uninstall', '-y', package],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False
        )

    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        '--quiet', '--no-cache-dir', 'opencv-python-headless==4.13.0.92'
    ])
    ready_file.touch()
    print('OpenCV is installed. Choose Runtime > Restart session, then choose Runtime > Run all.')
else:
    import cv2
    import numpy as np
    print('OpenCV:', cv2.__version__)


## 1. Choose the rule you will test

Start with the suggested value. Later, change only READY_TOLERANCE and observe how the decision changes.


In [ ]:
# Size of the blue target box in the middle of the frame.
TARGET_WIDTH = 360
TARGET_HEIGHT = 240
MIN_FACE_SIZE = 80

# Your first edit: how close must the face centre be to the target centre?
READY_TOLERANCE = 24

# OpenCV includes this pretrained face detector. It finds faces; it does not recognise identities.
face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

if face_detector.empty():
    raise RuntimeError('The face-detector file could not load. Restart the runtime and run from the top.')


## 2. Find one face box

The detector returns rectangles: x, y, width, height. We keep the largest visible face.


## 3. Draw the guide and choose a status

This helper draws the target, face box, centre points, and a clear message.


In [ ]:
def draw_alignment_guide(frame):
    output = frame.copy()
    frame_height, frame_width = output.shape[:2]
    left = (frame_width - TARGET_WIDTH) // 2
    top = (frame_height - TARGET_HEIGHT) // 2
    right = left + TARGET_WIDTH
    bottom = top + TARGET_HEIGHT
    target_centre = (left + TARGET_WIDTH // 2, top + TARGET_HEIGHT // 2)

    blue, cyan, orange, green = (21,138,194), (255,210,0), (246,138,69), (68,180,92)
    cv2.rectangle(output, (left, top), (right, bottom), cyan, 3)
    cv2.line(output, (frame_width // 2, top), (frame_width // 2, bottom), cyan, 2)
    cv2.drawMarker(output, target_centre, cyan, cv2.MARKER_CROSS, 20, 2)

    face_box = find_face_box(frame)
    status = 'FACE NOT FOUND — move into view'
    status_colour = orange
    if face_box is not None:
        x, y, width, height = face_box
        face_centre = (x + width // 2, y + height // 2)
        distance = ((face_centre[0] - target_centre[0]) ** 2 +
                    (face_centre[1] - target_centre[1]) ** 2) ** 0.5
        ready = left <= face_centre[0] <= right and top <= face_centre[1] <= bottom and distance <= READY_TOLERANCE
        cv2.rectangle(output, (x, y), (x + width, y + height), blue, 3)
        status_colour = green if ready else orange
        cv2.circle(output, face_centre, 8, status_colour, -1)
        cv2.line(output, target_centre, face_centre, status_colour, 2)
        status = 'READY — capture frame' if ready else 'ADJUST — centre your face'

    cv2.rectangle(output, (14, 14), (frame_width - 14, 60), (18, 25, 34), -1)
    cv2.putText(output, status, (26, 45), cv2.FONT_HERSHEY_SIMPLEX, .65, status_colour, 2)
    cv2.putText(output, f'tolerance: {READY_TOLERANCE}px', (18, frame_height - 18),
                cv2.FONT_HERSHEY_SIMPLEX, .6, (255,255,255), 2)
    return output, status


## 4A. Colab route — capture from your webcam

Run the next cell. Allow browser camera access, centre your face, then press CAPTURE FRAME. Run it again whenever you want to test a new frame.


In [ ]:
# This adapter opens the browser webcam after the learner clicks START CAMERA.
# You do not need to edit this cell.
from base64 import b64decode
from IPython.display import Javascript, display
from google.colab.output import eval_js

def capture_webcam_frame():
    display(Javascript(r'''
      async function captureFrame() {
        const panel = document.createElement('div');
        const start = document.createElement('button');
        const capture = document.createElement('button');
        const message = document.createElement('p');
        const video = document.createElement('video');

        start.textContent = 'START CAMERA';
        capture.textContent = 'CAPTURE FRAME';
        capture.disabled = true;
        start.style.cssText = capture.style.cssText =
          'padding:10px 16px;font-weight:bold;margin:8px 8px 8px 0;';
        video.style.cssText =
          'display:block;width:420px;max-width:100%;border:2px solid #10151a;';
        message.textContent = 'Click START CAMERA, then choose Allow in the browser prompt.';

        panel.appendChild(message);
        panel.appendChild(start);
        panel.appendChild(capture);
        panel.appendChild(video);
        document.body.appendChild(panel);

        // The browser allows camera access only after the learner clicks this button.
        const stream = await new Promise((resolve, reject) => {
          start.onclick = async () => {
            try {
              const webcam = await navigator.mediaDevices.getUserMedia({video: true});
              video.srcObject = webcam;
              await video.play();
              start.disabled = true;
              capture.disabled = false;
              message.textContent = 'Camera is live. Centre your face, then click CAPTURE FRAME.';
              resolve(webcam);
            } catch (error) {
              message.textContent = 'Camera permission was denied. Use your browser site settings to allow the camera, then run this cell again.';
              reject(error);
            }
          };
        });

        // Wait until the learner chooses the exact frame to analyse.
        await new Promise(resolve => capture.onclick = resolve);
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getTracks().forEach(track => track.stop());
        panel.remove();
        return canvas.toDataURL('image/jpeg');
      }
    '''))

    try:
        image_data = eval_js('captureFrame()')
    except Exception:
        # The browser message above explains how to allow the camera.
        return None

    image_bytes = b64decode(image_data.split(',')[1])
    return cv2.imdecode(np.frombuffer(image_bytes, np.uint8), cv2.IMREAD_COLOR)

# Start the webcam, position an face, then click CAPTURE FRAME.
webcam_frame = capture_webcam_frame()

if webcam_frame is not None:
    result, status = draw_alignment_guide(webcam_frame)
    from google.colab.patches import cv2_imshow
    cv2_imshow(result)
    print(status)
else:
    print('No frame was captured. Run this cell again after allowing camera access.')


## 4B. Local route — continuous live webcam

Run this only on your own computer, in a local Python or Jupyter environment. Press **q** in the camera window to stop.

In [ ]:
camera = cv2.VideoCapture(0)

if not camera.isOpened():
    print('No local webcam was found. In Colab, use the webcam-capture cell above.')
else:
    while True:
        ok, frame = camera.read()
        if not ok:
            break
        result, status = draw_alignment_guide(frame)
        cv2.imshow('Face framing guide — press q to exit', result)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    camera.release()
    cv2.destroyAllWindows()


## 5. Record evidence

1. With your face centred, what status appears?
2. With it off-centre, what status appears?
3. When it is out of view, what status appears?
4. After changing READY_TOLERANCE, does the guide approve too early, too late, or at a useful point?

Exit evidence: write one rule you changed, one visible result, and one case where a user could still be confused.
